# 00 — 数据下载 & Qlib 同步

**职责：** 下载原始行情 CSV，同步到 Qlib Provider，输出 `data/session_config.json` 供后续所有 notebook 共享。

```
00 下载+同步  →  01 因子研究  →  end_to_end 训练  →  02 信号验证  →  03 Spread 研究  →  05 注册管理
              ↑ 共享 session_config.json 和 data/csv_source/
```

**完成标志：** 最后一个 Cell 打印 `✅ SESSION READY`，生成 `data/session_config.json`。

## ⚙️ 全局配置 — 修改这里，其余 Cell 自动适配

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ⚙️  SESSION CONFIG — 只改这里，所有 notebook 共享
# ═══════════════════════════════════════════════════════════════
MARKET       = 'us'           # us / cn / hk
SYMBOLS      = [              # 你的股票池
    'AAPL', 'NVDA', 'MSFT', 'GOOGL', 'AMZN',
    'META', 'TSLA', 'AVGO', 'COST', 'NFLX',
]
BENCHMARK    = 'QQQ'          # us=QQQ / cn=000300 / hk=^HSI
TRAIN_START  = '2021-01-01'
TRAIN_END    = '2024-12-31'
TEST_START   = '2025-01-01'
TEST_END     = '2026-06-18'
# ═══════════════════════════════════════════════════════════════


In [ ]:
import sys, json, warnings
from pathlib import Path
import pandas as pd
warnings.filterwarnings('ignore')

ROOT = Path.cwd()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.common.qlib_init import build_qlib_init_cfg, safe_qlib_init
from src.data.router import MarketDataRouter
from src.data.adapters.yfinance_adapter import YFinanceAdapter
from src.data.adapters.efinance_adapter import EFinanceAdapter
from src.data.adapters.akshare_adapter import AkShareAdapter
from src.data.adapters.baostock_adapter import BaoStockAdapter
from src.data.validation.schema import validate_market_data
from src.data.quality import generate_data_quality_summary

CSV_DIR = ROOT / 'data' / 'csv_source'
CSV_DIR.mkdir(parents=True, exist_ok=True)
print(f'Root: {ROOT}')
print(f'Market: {MARKET.upper()}  Symbols: {len(SYMBOLS)}  Benchmark: {BENCHMARK}')
print(f'Train: {TRAIN_START} → {TRAIN_END}')
print(f'Test:  {TEST_START} → {TEST_END}')

## 1. 数据下载 — MarketDataRouter

In [ ]:
import matplotlib.pyplot as plt

router = MarketDataRouter(
    adapters=[YFinanceAdapter(), EFinanceAdapter(), AkShareAdapter(), BaoStockAdapter()],
    policy={'us': ['yfinance'], 'cn': ['efinance', 'akshare', 'baostock'], 'hk': ['yfinance']},
)

all_symbols = list(dict.fromkeys(SYMBOLS + [BENCHMARK]))  # benchmark 也要下载
results = {'ok': [], 'failed': [], 'schema_rejected': []}

for sym in all_symbols:
    csv_path = CSV_DIR / f'{sym}.csv'
    start = TRAIN_START
    existing = None
    if csv_path.exists():
        try:
            existing = pd.read_csv(csv_path); existing['date'] = pd.to_datetime(existing['date'])
            start = (existing['date'].max() - pd.Timedelta(days=30)).strftime('%Y-%m-%d')
        except Exception: pass

    resp = router.fetch_daily_bars(symbol=sym, market=MARKET, start=start, validate=True)
    if not resp.ok or resp.result is None:
        errs = '; '.join([f'{a.provider}: {a.error}' for a in resp.attempts if not a.ok])
        results['failed'].append((sym, errs)); print(f'  ❌ {sym}: {errs}'); continue

    ok, vdf, errs = validate_market_data(resp.result.df, sym)
    if not ok:
        results['schema_rejected'].append((sym, errs)); print(f'  ⚠️ {sym}: schema error'); continue

    if existing is not None:
        vdf = pd.concat([existing, vdf], ignore_index=True).drop_duplicates('date', keep='last').sort_values('date')
    vdf.to_csv(csv_path, index=False)
    results['ok'].append(sym)
    print(f'  ✅ {sym}: {len(vdf)} rows  {vdf["date"].min().date()} → {vdf["date"].max().date()}')

print(f'\n下载完成: ✅{len(results["ok"])}  ❌{len(results["failed"])}  ⚠️{len(results["schema_rejected"])}')

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(['OK', 'Failed', 'Schema Rej'],
       [len(results['ok']), len(results['failed']), len(results['schema_rejected'])],
       color=['#22c55e', '#ef4444', '#f59e0b'])
ax.set_title('Download Results'); plt.tight_layout(); plt.show()

## 2. Qlib 同步 — CSV → Provider

In [ ]:
import subprocess

QLIB_DIR = ROOT / 'data' / 'watchlist'
QLIB_DIR.mkdir(parents=True, exist_ok=True)

# Qlib dump_bin: 增量同步 CSV → Provider
cmd = [
    'python', '-m', 'qlib.run.dump_bin',
    '--csv_path', str(CSV_DIR),
    '--qlib_dir', str(QLIB_DIR),
    '--freq', 'day',
    '--date_field_name', 'date',
    '--symbol_field_name', 'symbol',
    '--include_fields', 'open,high,low,close,volume',
    '--mode', 'incremental',
]
print('Running: ' + ' '.join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(ROOT))
if result.returncode == 0:
    print('✅ Qlib sync OK')
else:
    print('⚠️ Qlib sync returned non-zero (may be OK for incremental mode):')
    print(result.stderr[-2000:] if result.stderr else '(no stderr)')

## 3. 三市场数据覆盖检查

In [ ]:
safe_qlib_init(build_qlib_init_cfg(None, market=MARKET))
from qlib.data import D

print(f'=== Qlib Provider Coverage ({MARKET.upper()}) ===')
coverage = {}
for sym in all_symbols:
    try:
        df = D.features([sym], ['$close'], start_time=TRAIN_START, end_time=TEST_END)
        n = len(df)
        last = str(df.index.get_level_values('datetime').max())[:10]
        coverage[sym] = {'rows': n, 'last': last, 'ok': n > 100}
        status = '✅' if n > 100 else '⚠️'
        print(f'  {status} {sym:8s}: {n:5d} rows  last={last}')
    except Exception as e:
        coverage[sym] = {'rows': 0, 'last': None, 'ok': False}
        print(f'  ❌ {sym}: {e}')

# CSV vs Qlib date alignment check
print('\n=== CSV vs Qlib date alignment ===')
mismatches = []
for sym in SYMBOLS:
    csv_path = CSV_DIR / f'{sym}.csv'
    if not csv_path.exists(): continue
    csv_last = pd.read_csv(csv_path, usecols=['date'])['date'].max()
    qlib_last = coverage.get(sym, {}).get('last')
    match = '✅' if qlib_last and csv_last[:10] == qlib_last else '⚠️'
    if match == '⚠️': mismatches.append(sym)
    print(f'  {match} {sym:8s}: CSV={csv_last[:10]}  Qlib={qlib_last}')

if mismatches:
    print(f'\n⚠️ Mismatched symbols (re-run sync): {mismatches}')
else:
    print('\n✅ All symbols aligned')

## 4. 数据质量汇总

In [ ]:
q = generate_data_quality_summary(
    dataset_key='watchlist', freq='day',
    provider_uri=QLIB_DIR, csv_dir=CSV_DIR, markets=[MARKET],
)
mkt = q.get('markets', {}).get(MARKET, {})
print(f"Quality: {'✅ OK' if q.get('ok') else '❌ FAILED'}  "      f"instruments={mkt.get('instruments','?')}  stale={mkt.get('stale_instruments',0)}  "      f"csv_missing={mkt.get('csv_missing',0)}")
for w in q.get('warnings', [])[:5]:
    print(f'  ⚠️ {w}')

## 5. 保存 session_config.json → 供后续 notebook 读取

In [ ]:
from datetime import datetime

session_cfg = {
    'market': MARKET,
    'symbols': SYMBOLS,
    'benchmark': BENCHMARK,
    'train_start': TRAIN_START,
    'train_end': TRAIN_END,
    'test_start': TEST_START,
    'test_end': TEST_END,
    'created_at': datetime.now().isoformat(),
    'download_ok': results['ok'],
    'download_failed': results['failed'],
}

cfg_path = ROOT / 'data' / 'session_config.json'
cfg_path.write_text(json.dumps(session_cfg, indent=2))
print(f'✅ SESSION READY → {cfg_path}')
print('─' * 60)
print('下一步: 打开 01_factor_research.ipynb 或 end_to_end_training_pipeline.ipynb')
print('  它们会自动读取 session_config.json，无需重新配置。')